In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/spotify_2015_2025_85k.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


release_col = "release_date"
genre_col = "genre"
stream_col = "stream_count"


if release_col == "release_date":
    df["year"] = pd.to_datetime(df[release_col], errors="coerce").dt.year
else:
    df["year"] = pd.to_numeric(df[release_col], errors="coerce")


df = df[["year", genre_col, stream_col]].copy()
df = df.rename(columns={
    genre_col: "genre",
    stream_col: "stream_count"
})


df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["stream_count"] = pd.to_numeric(df["stream_count"], errors="coerce")
df["genre"] = df["genre"].astype(str).str.strip()

df = df.dropna(subset=["year", "genre", "stream_count"])
df = df[df["genre"] != ""]
df = df[~df["genre"].str.lower().isin(["unknown", "none", "nan"])]
df = df[df["year"].between(2015, 2025)]


top3_genres = (
    df.groupby("genre", as_index=False)["stream_count"]
    .mean()
    .sort_values("stream_count", ascending=False)
    .head(3)["genre"]
    .tolist()
)


df_top3 = df[df["genre"].isin(top3_genres)].copy()


df_plot = (
    df_top3.groupby(["year", "genre"], as_index=False)["stream_count"]
        .mean()
        .sort_values(["year", "genre"])
)


color_map = {
    top3_genres[0]: "#E29C05",  
    top3_genres[1]: "#B61B82",  
    top3_genres[2]: "#5E6CEC"   
}


fig = px.line(
    df_plot,
    x="year",
    y="stream_count",
    color="genre",
    markers=False,
    color_discrete_map=color_map,
    category_orders={"genre": top3_genres},
    title="Evolución del consumo promedio de los 3 géneros más populares (2015–2025)",
    labels={
        "year": "Año",
        "stream_count": "Consumo promedio (stream count)",
        "genre": "Género"
    }
)


fig.update_traces(
    line_shape="spline",
    line=dict(width=4),
    marker=dict(size=8),
    hoverinfo="none"
)


fig.update_yaxes(
    range=[0, df_plot["stream_count"].max() * 1.08],
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)",
    zeroline=False,
    title_font=dict(size=14),
    tickformat="~s"   # 50k, 100k, etc. 
)

fig.update_xaxes(
    tickmode="linear",
    dtick=1,
    showgrid=False,
    zeroline=False,
    title_font=dict(size=14)
)


fig.update_layout(
    template="simple_white",
    width=1100,
    height=650,

    title=dict(
        text="Evolución del consumo promedio de los 3 géneros más populares (2015–2025)",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),

    legend=dict(
        title="Género",
        orientation="h",
        yanchor="bottom",
        y=1,
        xanchor="center",
        x=0.5
    ),

    margin=dict(l=80, r=40, t=120, b=110),
    plot_bgcolor="white",
    paper_bgcolor="white",

    font=dict(
        family="Arial, sans-serif",
        size=13
    )
)

# =========================================
# ANOTACIONES GENERALES (EDITABLES)
# =========================================

# Subtitle under title
fig.add_annotation(
    text="Datos basados en el promedio anual de streams por género a nivel global.",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.12,
    showarrow=False,
    align="center",
    font=dict(size=14, color="gray")
)

# Bottom note
fig.add_annotation(
    text=(
        "Cada línea representa el consumo promedio anual, no valores acumulados.<br>"
        "Fuente: Spotify Music Analytics Dataset (2015–2025) / Rohit kumar"
    ),
    xref="paper",
    yref="paper",
    x=0,
    y=-0.22,
    showarrow=False,
    align="left",
    font=dict(size=11, color="gray")
)

max_min_points = []

for genre in top3_genres:
    subset = df_plot[df_plot["genre"] == genre].copy()
    
    max_row = subset.loc[subset["stream_count"].idxmax()]
    min_row = subset.loc[subset["stream_count"].idxmin()]
    genre_color = color_map[genre]
    
    max_min_points.append({
        "year": max_row["year"],
        "stream_count": max_row["stream_count"],
        "genre": genre,
        "type": "max"
    })
    max_min_points.append({
        "year": min_row["year"],
        "stream_count": min_row["stream_count"],
        "genre": genre,
        "type": "min"
    })



for point in max_min_points:
    genre = point["genre"]
    genre_color = color_map[genre]
    
    fig.add_scatter(
        x=[point["year"]],
        y=[point["stream_count"]],
        mode="markers",
        marker=dict(
            size=16,
            color="rgba(0,0,0,0)",  
            line=dict(
                color=genre_color,
                width=3
            )
        ),
        showlegend=False,
        hoverinfo="none"
    )



genre_0_max = [p for p in max_min_points if p["type"] == "max" and p["genre"] == top3_genres[0]][0] if any(p["type"] == "max" and p["genre"] == top3_genres[0] for p in max_min_points) else None
genre_0_min = [p for p in max_min_points if p["type"] == "min" and p["genre"] == top3_genres[0]][0] if any(p["type"] == "min" and p["genre"] == top3_genres[0] for p in max_min_points) else None
genre_1_max = [p for p in max_min_points if p["type"] == "max" and p["genre"] == top3_genres[1]][0] if any(p["type"] == "max" and p["genre"] == top3_genres[1] for p in max_min_points) else None
genre_1_min = [p for p in max_min_points if p["type"] == "min" and p["genre"] == top3_genres[1]][0] if any(p["type"] == "min" and p["genre"] == top3_genres[1] for p in max_min_points) else None
genre_2_max = [p for p in max_min_points if p["type"] == "max" and p["genre"] == top3_genres[2]][0] if any(p["type"] == "max" and p["genre"] == top3_genres[2] for p in max_min_points) else None
genre_2_min = [p for p in max_min_points if p["type"] == "min" and p["genre"] == top3_genres[2]][0] if any(p["type"] == "min" and p["genre"] == top3_genres[2] for p in max_min_points) else None


if genre_0_max:
    fig.add_annotation(
        x=genre_0_max["year"],
        y=genre_0_max["stream_count"],
        text=(
            "Artistas destacados<br>"
            "Viralidad redes sociales<br>"
            "(trends, memes)"
        ),
        showarrow=False,
        bgcolor="white",
        bordercolor=color_map[top3_genres[0]],
        borderwidth=1.5,
        font=dict(size=11, color=color_map[top3_genres[0]]),
        xshift=-15,
        yshift=45,
        align="left",
        textangle=0,
        valign="top"
    )




if genre_1_max:
    fig.add_annotation(
        x=genre_1_max["year"],
        y=genre_1_max["stream_count"],
        text=(
            "Nostalgia 90s/2000s<br>"
            "Nuevos artistas<br>"
            "Viralidad en TikTok"
        ),
        showarrow=False,
        bgcolor="white",
        bordercolor=color_map[top3_genres[1]],
        borderwidth=1.5,
        font=dict(size=11, color=color_map[top3_genres[1]]),
        xshift=25,
        yshift=45,
        align="left",
        textangle=0,
        valign="top"
    )



if genre_2_max:
    fig.add_annotation(
        x=genre_2_max["year"],
        y=genre_2_max["stream_count"],
        text=(
            "Resurgimiento del metal:<br>"
            "Álbumes destacados, redes sociales<br>"
            "y 'Metal lords' acercando nuevo público"
        ),
        showarrow=False,
        bgcolor="white",
        bordercolor=color_map[top3_genres[2]],
        borderwidth=1.5,
        font=dict(size=11, color=color_map[top3_genres[2]]),
        xshift=-45,
        yshift=35,
        align="left",
        textangle=0,
        valign="top"
    )



for genre in top3_genres:
    subset = df_plot[df_plot["genre"] == genre].copy()
    genre_color = color_map[genre]
    
    years = sorted(subset["year"].unique())
    
    for year in years:
        if year != 2022:
            # Check if this is a max or min point
            is_max = any(p["year"] == year and p["genre"] == genre and p["type"] == "max" for p in max_min_points)
            is_min = any(p["year"] == year and p["genre"] == genre and p["type"] == "min" for p in max_min_points)
            
            # Only add circle if it's NOT a max or min point
            if not is_max and not is_min:
                year_data = subset[subset["year"] == year]
                if not year_data.empty:
                    fig.add_scatter(
                        x=year_data["year"],
                        y=year_data["stream_count"],
                        mode="markers",
                        marker=dict(
                            size=8,
                            color=genre_color
                        ),
                        showlegend=False,
                        hoverinfo="none"
                    )


fig.show()


fig.write_html("index.html", include_plotlyjs="cdn")